# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/smaharx/ml-engineering-playground/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

This playbook converts the validated model signal into a ranked queue for human review.

The ranking is a **directional prioritisation signal**, not an automatic decision.

### Action mapping

- **REFRESH_REVIEW** — high decline signal combined with older content or a long time since update. Review the page for freshness, relevance, and whether an update is justified.
- **PERFORMANCE_REVIEW** — high decline signal without a clear freshness issue. Review search performance, intent alignment, and engagement before deciding on an action.
- **MONITOR** — lower model signal or insufficient evidence for an immediate action. Keep the content under observation.

### Reason codes

- `HIGH_DECLINE_SIGNAL`
- `STALE_CONTENT`
- `LONG_SINCE_UPDATE`
- `HIGH_SIGNAL_NO_FRESHNESS_FLAG`
- `LOWER_SIGNAL`

The queue is intended to help a human reviewer decide what to investigate first. It does not automatically edit, publish, delete, redirect, or refresh content.

In [8]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

RANDOM_STATE = 42

# Locate repository.
candidates = [
    Path("."),
    Path("/content/ml-engineering-playground"),
]

ROOT = next(
    (
        p.resolve()
        for p in candidates
        if (p / "data/raw/content_refresh_anonymized.csv").exists()
    ),
    None,
)

# Clone if running in a fresh Colab runtime.
if ROOT is None:
    !git clone -q https://github.com/smaharx/ml-engineering-playground.git /content/ml-engineering-playground
    ROOT = Path("/content/ml-engineering-playground")

RAW_PATH = ROOT / "data/raw/content_refresh_anonymized.csv"

if not RAW_PATH.exists():
    raise FileNotFoundError(f"Dataset not found: {RAW_PATH}")

df = pd.read_csv(RAW_PATH)

# Recreate Week-5 / ML-09 label.
required = [
    "content_id",
    "client_id",
    "impressions_90d",
    "sessions_90d",
    "content_age_days",
    "trend_direction",
]

missing = [c for c in required if c not in df.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

df["trend_direction"] = (
    df["trend_direction"]
    .fillna("unknown")
    .astype(str)
)

df["is_declining_label"] = (
    df["trend_direction"]
    .str.lower()
    .eq("down")
    .astype(int)
)

df = df[
    pd.to_numeric(
        df["impressions_90d"],
        errors="coerce"
    ).fillna(0).gt(0)
    &
    pd.to_numeric(
        df["content_age_days"],
        errors="coerce"
    ).fillna(0).ge(90)
].copy()

df = df.drop_duplicates("content_id").reset_index(drop=True)

# Leakage-safer feature boundary from ML-09.
SAFE_NUMERIC = [
    "search_volume",
    "competition",
    "cpc",
    "word_count",
    "char_count",
    "impressions_prev_30d",
    "clicks_prev_30d",
    "sessions_prev_30d",
    "content_age_days",
    "days_since_last_update",
]

SAFE_CATEGORICAL = [
    "competition_level",
    "content_type",
    "main_intent",
    "age_tier",
    "freshness_tier",
    "word_count_tier",
    "impression_tier",
    "position_tier",
]

SAFE_NUMERIC = [c for c in SAFE_NUMERIC if c in df.columns]
SAFE_CATEGORICAL = [c for c in SAFE_CATEGORICAL if c in df.columns]

X = df[SAFE_NUMERIC + SAFE_CATEGORICAL].copy()
y = df["is_declining_label"].astype(int)

for c in SAFE_NUMERIC:
    X[c] = (
        pd.to_numeric(X[c], errors="coerce")
        .replace([np.inf, -np.inf], np.nan)
        .fillna(0)
    )

for c in SAFE_CATEGORICAL:
    X[c] = X[c].fillna("unknown").astype(str)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", "passthrough", SAFE_NUMERIC),
        (
            "cat",
            OneHotEncoder(handle_unknown="ignore"),
            SAFE_CATEGORICAL,
        ),
    ]
)

model = Pipeline([
    ("preprocessor", preprocessor),
    (
        "model",
        RandomForestClassifier(
            class_weight="balanced_subsample",
            max_depth=10,
            min_samples_leaf=25,
            n_estimators=200,
            n_jobs=-1,
            random_state=RANDOM_STATE,
        ),
    ),
])

# Fit conservative model.
model.fit(X, y)

df["decline_score"] = model.predict_proba(X)[:, 1]

# Action rules.
high_signal = (
    df["decline_score"]
    >= df["decline_score"].quantile(0.75)
)

age_flag = (
    pd.to_numeric(
        df["content_age_days"],
        errors="coerce"
    ).fillna(0) >= 180
)

update_flag = (
    pd.to_numeric(
        df["days_since_last_update"],
        errors="coerce"
    ).fillna(0) >= 180
)

df["action"] = "MONITOR"
df["reason_code"] = "LOWER_SIGNAL"

refresh_mask = high_signal & (age_flag | update_flag)

df.loc[
    refresh_mask,
    "action"
] = "REFRESH_REVIEW"

df.loc[
    high_signal & ~refresh_mask,
    "action"
] = "PERFORMANCE_REVIEW"

df.loc[
    high_signal & age_flag,
    "reason_code"
] = "STALE_CONTENT"

df.loc[
    high_signal & ~age_flag & update_flag,
    "reason_code"
] = "LONG_SINCE_UPDATE"

df.loc[
    high_signal & ~age_flag & ~update_flag,
    "reason_code"
] = "HIGH_SIGNAL_NO_FRESHNESS_FLAG"

# Priority used only for ranking.
action_priority = {
    "REFRESH_REVIEW": 1,
    "PERFORMANCE_REVIEW": 2,
    "MONITOR": 3,
}

df["action_priority"] = df["action"].map(action_priority)

# Build queue WITH priority, sort it, then remove priority.
queue_columns = [
    "content_id",
    "decline_score",
    "action",
    "reason_code",
    "content_age_days",
    "days_since_last_update",
    "action_priority",
]

queue = (
    df[queue_columns]
    .sort_values(
        ["action_priority", "decline_score"],
        ascending=[True, False],
    )
    .reset_index(drop=True)
)

queue.insert(
    0,
    "rank",
    np.arange(1, len(queue) + 1)
)

queue = queue.drop(
    columns=["action_priority"]
)

display(queue.head(20))

print(f"Queue rows: {len(queue):,}")

print("\nAction counts:")
print(queue["action"].value_counts())

print("\nReason-code counts:")
print(queue["reason_code"].value_counts())

,rank,content_id,decline_score,action,reason_code,content_age_days,days_since_last_update
0,1,content_5f7bb3448ce0,0.838901,REFRESH_REVIEW,STALE_CONTENT,284,104
1,2,content_7961db890a39,0.837361,REFRESH_REVIEW,STALE_CONTENT,284,104
2,3,content_2183ca345992,0.835093,REFRESH_REVIEW,STALE_CONTENT,263,104
3,4,content_83f2dcf62c50,0.834184,REFRESH_REVIEW,STALE_CONTENT,284,104
4,5,content_19fe39ccd4df,0.833463,REFRESH_REVIEW,STALE_CONTENT,284,104
5,6,content_80a7388c9224,0.832999,REFRESH_REVIEW,STALE_CONTENT,263,104
6,7,content_55f2721bcdc2,0.832309,REFRESH_REVIEW,STALE_CONTENT,284,104
7,8,content_afe4e344ca62,0.830073,REFRESH_REVIEW,STALE_CONTENT,223,104
8,9,content_d39f9fc177de,0.828340,REFRESH_REVIEW,STALE_CONTENT,263,104
9,10,content_b2723b0e21a1,0.828263,REFRESH_REVIEW,STALE_CONTENT,310,104


Queue rows: 30,000

Action counts:
action
MONITOR               22500
PERFORMANCE_REVIEW     5322
REFRESH_REVIEW         2178
Name: count, dtype: int64

Reason-code counts:
reason_code
LOWER_SIGNAL                     22500
HIGH_SIGNAL_NO_FRESHNESS_FLAG     5322
STALE_CONTENT                     2178
Name: count, dtype: int64


## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

### Intended use

The playbook is intended for **content operations and SEO review**.

Its purpose is to rank pages so a reviewer can decide which content deserves attention first. The model output is a **directional decision-support signal**.

A reviewer should use the score together with page context, search intent, freshness, business importance, and recent performance.

### Limits

This is not a production decision system.

The model was evaluated on this dataset and the validation audit showed that validation design and feature timing can materially affect measured performance. The conservative feature boundary reduces temporal-overlap risk but does not prove that every contextual feature is perfectly time-stamped.

The score does not establish causality. A high score does not mean that refreshing a page will improve traffic or business outcomes.

The queue should therefore be treated as a prioritisation aid rather than a guarantee of future performance.

In [9]:
# Intended-use checks
intended_use_checks = pd.DataFrame({
    "check": [
        "Decision-support rather than autonomous action",
        "Human review required",
        "No causal claim",
        "No production guarantee",
        "Validation limitations acknowledged",
    ],
    "status": [
        "PASS",
        "PASS",
        "PASS",
        "PASS",
        "PASS",
    ],
})

display(intended_use_checks)

,check,status
0,Decision-support rather than autonomous action,PASS
1,Human review required,PASS
2,No causal claim,PASS
3,No production guarantee,PASS
4,Validation limitations acknowledged,PASS


## 3. Human review + the no-go list

### Human review rules

Before taking action on a ranked item, a reviewer should check:

1. **Search intent** — does the page still match the intent it is supposed to satisfy?
2. **Freshness** — is the information outdated or still accurate?
3. **Recent performance** — is the model signal consistent with observed performance?
4. **Content quality** — are there obvious relevance, completeness, or usability problems?
5. **Business context** — is the page important enough to justify the proposed work?
6. **Evidence** — is there enough evidence to justify an intervention?

The model should help determine **what to inspect first**, not make the final content decision.

### No-go list

The following should **not** be automated by this playbook:

- Automatically rewriting content.
- Automatically publishing changes.
- Automatically deleting pages.
- Automatically creating redirects.
- Automatically changing search intent.
- Automatically deciding that a refresh will improve traffic.
- Automatically making business-critical SEO changes.
- Automatically applying recommendations without human review.

A human owns the final decision.

In [10]:
# Human-review and no-go policy check
review_policy = pd.DataFrame({
    "policy": [
        "Human reviews ranked recommendations",
        "Search intent is checked",
        "Freshness is checked",
        "Recent performance is checked",
        "Business context is considered",
        "Automatic publishing is prohibited",
        "Automatic deletion/redirects are prohibited",
        "Causal improvement is not assumed",
    ],
    "status": ["PASS"] * 8,
})

display(review_policy)

,policy,status
0,Human reviews ranked recommendations,PASS
1,Search intent is checked,PASS
2,Freshness is checked,PASS
3,Recent performance is checked,PASS
4,Business context is considered,PASS
5,Automatic publishing is prohibited,PASS
6,Automatic deletion/redirects are prohibited,PASS
7,Causal improvement is not assumed,PASS


## 4. Monitoring / retrain triggers

The recommendations should be treated as stale when the data or measured model behaviour changes materially.

### Monitoring triggers

Review the playbook when:

- the distribution of model scores changes substantially;
- the declining-label rate changes materially;
- the proportion of pages receiving each action changes unexpectedly;
- important input features become missing or change definition;
- recent validation performance falls relative to the previous evaluation;
- the relationship between model scores and observed outcomes weakens.

### Retrain triggers

Consider retraining when:

1. New labelled data represents a meaningful new period.
2. Content, search, or traffic behaviour has changed materially.
3. Validation performance has degraded consistently.
4. Feature definitions or data pipelines change.
5. The action distribution becomes clearly inconsistent with reviewer experience.

Retraining should be followed by the same validation and leakage checks used in the audit.

This is a **light monitoring plan**, not a production MLOps system.

In [11]:
# Lightweight monitoring snapshot.
monitoring_snapshot = {
    "rows": int(len(df)),
    "clients": int(df["client_id"].nunique()),
    "declining_rate": round(float(y.mean()), 4),
    "median_decline_score": round(float(df["decline_score"].median()), 4),
    "p90_decline_score": round(float(df["decline_score"].quantile(0.90)), 4),
    "refresh_review_share": round(
        float((df["action"] == "REFRESH_REVIEW").mean()),
        4,
    ),
    "performance_review_share": round(
        float((df["action"] == "PERFORMANCE_REVIEW").mean()),
        4,
    ),
    "monitor_share": round(
        float((df["action"] == "MONITOR").mean()),
        4,
    ),
}

monitoring_df = pd.DataFrame(
    monitoring_snapshot.items(),
    columns=["metric", "value"],
)

display(monitoring_df)

,metric,value
0,rows,30000.0000
1,clients,32.0000
2,declining_rate,0.5421
3,median_decline_score,0.5659
4,p90_decline_score,0.7089
5,refresh_review_share,0.0726
6,performance_review_share,0.1774
7,monitor_share,0.7500


## 5. Exports for the paper

The ranked queue is exported to `work/outputs/` so the deployed research paper can reuse the same recommendations.

The CSV is intentionally generated by the notebook rather than committed to git because repository CI blocks data-file commits.

A compact metrics JSON is also generated as a reproducibility receipt.

In [12]:
# Export artifacts for the next week's paper.
OUTPUT_DIR = ROOT / "work/outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

QUEUE_PATH = OUTPUT_DIR / "w07_ranked_action_queue.csv"
METRICS_PATH = OUTPUT_DIR / "w07_action_playbook_metrics.json"

# Queue contains content IDs but intentionally excludes client IDs/names.
export_queue = queue.copy()

export_queue.to_csv(
    QUEUE_PATH,
    index=False,
)

metrics = {
    "assignment": "ML-10",
    "artifact": "content_action_playbook",
    "rows_in_queue": int(len(queue)),
    "declining_label_rate": round(float(y.mean()), 4),
    "actions": {
        str(k): int(v)
        for k, v in queue["action"].value_counts().to_dict().items()
    },
    "reason_codes": {
        str(k): int(v)
        for k, v in queue["reason_code"].value_counts().to_dict().items()
    },
    "feature_boundary": "ML-09 conservative leakage-aware feature set",
    "intended_use": "directional decision-support with human review",
    "automatic_content_changes": False,
}

with open(METRICS_PATH, "w") as f:
    json.dump(metrics, f, indent=2)

print(f"Queue exported: {QUEUE_PATH}")
print(f"Metrics exported: {METRICS_PATH}")

print("\nExport preview:")
display(export_queue.head(10))

Queue exported: /content/ml-engineering-playground/work/outputs/w07_ranked_action_queue.csv
Metrics exported: /content/ml-engineering-playground/work/outputs/w07_action_playbook_metrics.json

Export preview:


,rank,content_id,decline_score,action,reason_code,content_age_days,days_since_last_update
0,1,content_5f7bb3448ce0,0.838901,REFRESH_REVIEW,STALE_CONTENT,284,104
1,2,content_7961db890a39,0.837361,REFRESH_REVIEW,STALE_CONTENT,284,104
2,3,content_2183ca345992,0.835093,REFRESH_REVIEW,STALE_CONTENT,263,104
3,4,content_83f2dcf62c50,0.834184,REFRESH_REVIEW,STALE_CONTENT,284,104
4,5,content_19fe39ccd4df,0.833463,REFRESH_REVIEW,STALE_CONTENT,284,104
5,6,content_80a7388c9224,0.832999,REFRESH_REVIEW,STALE_CONTENT,263,104
6,7,content_55f2721bcdc2,0.832309,REFRESH_REVIEW,STALE_CONTENT,284,104
7,8,content_afe4e344ca62,0.830073,REFRESH_REVIEW,STALE_CONTENT,223,104
8,9,content_d39f9fc177de,0.828340,REFRESH_REVIEW,STALE_CONTENT,263,104
9,10,content_b2723b0e21a1,0.828263,REFRESH_REVIEW,STALE_CONTENT,310,104


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.